In [44]:
#创建数据库引擎管理模块（db_manager.py） 
import os
import sys
sys.path.append(r'D:\pyprj\cv2\utils')
from JSONConfig import ConfigHandler
import pandas as pd 
import mysql.connector
from mysql.connector import errorcode 
from queue import Queue, Empty
from contextlib import contextmanager 
from pathlib import Path

#获取当前Notebook的绝对路径
# current_dir=Path(file).resolve()  
# project_root= current_dir.parent.parent
class MySQLHelper:
    _instance = None #单例模
    def __new__(cls,*args,**kwargs):
        if cls._instance is None:
            cls._instance = super(MySQLHelper,cls).__new__(cls)
        return cls._instance
    def __init__(self,config_path=r'D:\pyprj\cv2\utils\config.json',db='ginger2'):
        self.config_path = config_path
        if getattr(self,'db',None) != db:
            self.db=db
            cfghandle = ConfigHandler(self.config_path)
            cfg =cfghandle.config_data[self.db]
            cfg['password']=cfghandle.get_decrypt((self.db,'password'))
            self.host = cfg['host']
            self.port = cfg['port']
            self.user = cfg['username']
            self.password = cfg['password']
            self.database = cfg['dbname']
            self.max_connections = cfg['maxconnection']
            self.connections = Queue(maxsize=self.max_connections)
            #self.initialized=True
            self._initialize_connections()
    def _initialize_connections(self):
            for _ in range(3): #Default to create 3 connections 
                self._create_connection()
    def _create_connection(self):
        if self.connections.qsize() < self.max_connections: 
            try:
                connection = mysql.connector.connect(
                    host=self.host, 
                    port=self.port,
                    auth_plugin='mysql_native_password',
                    user=self.user,
                    password=self.password, 
                    database=self.database
                )
                self.connections.put(connection)
            except mysql.connector.Error as err:
                print("Error creating connection:",err)
    def _get_connection(self):
        try:
            connection = self.connections.get(timeout=5) #Wait up to 5 seconds for a connection 
            if not connection.is_connected():
                connection.reconnect(attempts=3,delay=5)
            return connection 
        except Empty:
            self._create_connection()
            return self._get_connection()
    def _release_connection(self,connection): 
        if connection.is_connected():
            if self.connections.qsize() < self.max_connections:
                self.connections.put(connection)
            else:
                connection.close()
                print("closed extra connection")
        else:
            self._create_connection()

    @contextmanager
    def connection_manager(self):
        connection = self._get_connection()
        try:
            yield connection 
        finally:
            self._release_connection(connection)
    def close_all(self):
        while not self.connections.empty(): 
            connection = self.connections.get()
            connection.close()
        print("All connections closed")
    def insert_single(self,table,data):
        with self.connection_manager() as connection: 
            cursor = connection.cursor()
            placeholders =','.join(['%s'] * len(data))
            columns =','.join(data.keys() )
            sql =f"INSERT INTO {table} ({columns}) VALUES ({placeholders})"
            cursor.execute(sql, list(data.values())) 
            connection.commit()
            print("Insert successful") 
            cursor.close()
    def bulk_insert(self,table,df):
        with self.connection_manager() as connection:
            cursor = connection.cursor()
            placeholders = ','.join(['%s'] * len(df.columns))
            columns =','.join(df.columns)
            sql=f"INSERT INTO {table}({columns}) VALUES({placeholders})" 
            data =[tuple(x) for x in df.values]
            cursor.executemany(sql,data)
            connection.commit()
            print("Bulk insert successful")
            cursor.close()
    def query(self,sql,dct=True):
        with self.connection_manager() as connection:
            cursor = connection.cursor(dictionary=dct)
            cursor.execute(sql)
            result = cursor.fetchall()
            cursor.close()
            return result
    def query2df(self,sql):
        with self.connection_manager() as connection:
            df = pd.read_sql(sql, connection) 
            return df
    def qry2df(self, sql):
        with self.connection_manager() as connection:
            cursor = connection.cursor() 
            cursor.execute(sql)
            columns =[desc[0] for desc in cursor.description]
            data = cursor.fetchall()
            cursor.close()
            df = pd.DataFrame(data,columns=columns)
            return df
    def bulk_update(self,table,df,key):
        with self.connection_manager() as connection: 
            cursor = connection.cursor()
            for index,row in df.iterrows():
                set_clause=','.join([f"{col} = %s" for col in df.columns if col != key])
                sq1=f"UPDATE {table} SET {set_clause} WHERE {key} = %s"
                values=[row[col] for col in df.columns if col !=key] + [row[key]]
                cursor.execute(sql,values)
            connection.commit()
            print("Bulk update successful")
            cursor.close()
    def delete(self,table,condition):
        with self.connection_manager() as connection:
            cursor = connection.cursor()
            sql=f"DELETE FROM {table} WHERE {condition}"
            cursor.execute(sql) 
            connection.commit()
            print("Delete successful")
            cursor.close()
    def truncate(self,table):
        self.close_all()
        with self.connection_manager() as connection:
            cursor = connection.cursor()
            sql = f"TRUNCATE TABLE {table}"
            cursor.execute(sql)
            connection.commit()
            print(f"Table {table} truncated successfully")
            cursor.close()
            connection.close()
        #Reinitialize connections after truncating 
        self._initialize_connections()

    

In [45]:

if __name__=="__main__":
    #初始化时自动加载（Jupyter首次导入时执行）
    db = MySQLHelper() #插入单条记录
    print(db.host)
    # data ={'id':9001,'1v5':'测试1'} 
    # db.insert_single('ts_import',data)
    #批量插入
    # df=pd.DataFrame({['id':[9002,9003],'1v5':['测试2','测i试3']})
    # db.bulk_insert('ts_import',df)
    #查询数据：#查询
    result = db.query('select * from ginger2.book',True)
    print(result)
    
    #批量更新
    # df_update=pd.DataFrame({'id':[1,2],'col1':['new_value1','new_value3'],'co12':['new_value2','new_value4']})
    # db.bulk_update('my_table',df_update,key='id')

127.0.0.1
[{'create_time': 1608114840, 'status': 1, 'id': 1, 'title': 'python高级编程', 'author': 'kim', 'binding': '精装', 'publisher': '清华大学出版社', 'price': '85.00', 'pages': 385, 'pubdate': '2020-12-18', 'isbn': '1234542445677', 'summary': '这是一本非常牛叉的书籍', 'image': None}, {'create_time': 1608114840, 'status': 1, 'id': 2, 'title': 'JavaScript特级编程', 'author': 'kim', 'binding': '精装', 'publisher': '机械工业出版社', 'price': '185', 'pages': 585, 'pubdate': '2020-12-20', 'isbn': '8765678765432', 'summary': '这是一本特级牛叉的书', 'image': None}, {'create_time': 1608114840, 'status': 1, 'id': 3, 'title': '华为的冬天', 'author': 'kim', 'binding': '精装', 'publisher': '华为出版社', 'price': '187', 'pages': 585, 'pubdate': '2020-12-20', 'isbn': '1982736789875', 'summary': '这是虚构的书籍', 'image': ''}]


In [43]:
db._instance = None

In [ ]:
mysql.connector.connect(
                    host='127.0.0.1', 
                    port=3308,
                    user='root',
                    password='kim789', 
                    # auth_plugin='mysql_native_password',
                    database='ginger2' #,use_pure=True,ssl_disabled= True
                )

In [ ]:
import mysql.connector
from mysql.connector import Error

try:
    # 完全复制 Navicat 的参数
    conn = mysql.connector.connect(
        host='127.0.0.1',  # 优先用 127.0.0.1 替代 localhost（Navicat 若用这个，必须一致）
        port=3308,         # 严格和 Navicat 端口一致
        user='root',       # Navicat 的用户名
        password='kim789',  # 注意：空密码填 ''，不要留空或不写
        database='ginger2',   # Navicat 中选中的默认数据库（可选，但建议加）
        auth_plugin='mysql_native_password'  # 关键！解决 MySQL 8.0 认证问题
    )
    if conn.is_connected():
        print("连接成功！")
        conn.close()
except Error as e:
    print(f"连接失败：{str(e)}")  # 打印完整错误，方便定位